# **Proyek ETL Pipeline & Analisis Pemasaran Perbankan (Bank Marketing)**
## **Layer 3: Gold Layer (Business Aggregations, Data Mart & Pipeline Operationalization)**

Notebook ini merupakan tahap puncak dari arsitektur Medallion. Pada tahap ini dilakukan:
1. **Membaca Data dari Silver:** Mengambil data bersih dan ter-enrich dari `lakehouse/silver/bank_marketing`.
2. **Business Aggregations & Data Mart Creation:** Membangun tabel agregasi analitik bisnis yang siap dikonsumsi oleh Business Intelligence (BI) dan analis pemasaran.
3. **Penyimpanan ke Gold Layer & Ekspor:** Menyimpan data mart ke format Parquet (`lakehouse/gold/bank_marketing_summary`) dan CSV tunggal.
4. **Panduan Operasional & Otomatisasi Pipeline:** Dokumentasi lengkap implementasi Jobs, DAG dependensi, retry policy, penjadwalan, dan monitoring pipeline.

---
### **1. Inisialisasi Sesi Spark**

In [ ]:
# Instal pyspark jika dijalankan di Google Colab terpisah
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, round, col

# Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("BankMarketing_03_Gold") \
    .getOrCreate()

print("Spark Session untuk Layer Gold berhasil diinisialisasi.")

---
### **2. Membaca Data dari Layer Silver**

In [ ]:
silver_path = "lakehouse/silver/bank_marketing"

# Ambil data dari Silver
silver_data = spark.read.parquet(silver_path)

print(f"Data Silver berhasil dimuat dari: {silver_path}")
print(f"Total baris data Silver: {silver_data.count()}")
silver_data.printSchema()
silver_data.show(5)

---
### **3. Agregasi Bisnis (Business Data Mart: AgeGroup & Target Deposito)**

Membangun ringkasan tingkat bisnis untuk menganalisis performa kampanye berdasarkan segmen usia dan keputusan berlangganan deposito (`y`):
* `total_klien`: Total jumlah prospek pada segmen terkait.
* `avg_saldo`: Rata-rata saldo tahunan nasabah.
* `avg_durasi_telepon`: Rata-rata durasi percakapan telepon kampanye (dalam detik).

In [ ]:
# Buat agregasi ringkas sebagai data mart Gold
gold_agg = silver_data.groupBy("AgeGroup", "y") \
    .agg(
        count("*").alias("total_klien"),
        round(avg("balance"), 2).alias("avg_saldo"),
        round(avg("duration"), 2).alias("avg_durasi_telepon")
    ) \
    .orderBy("AgeGroup", "y")

print("=== Tabel Data Mart Gold (Summary Kampanye Bank Marketing) ===")
gold_agg.show()

---
### **4. Penyimpanan ke Gold Layer & Ekspor CSV Tunggal**

In [ ]:
# Simpan ke Gold Layer (format Parquet)
gold_path = "lakehouse/gold/bank_marketing_summary"
gold_agg.write.mode("overwrite").parquet(gold_path)

print(f"Layer Gold Berhasil Terbentuk di: {gold_path}")

# Verifikasi membaca kembali data Gold
print("\nVerifikasi Data Gold dari Parquet:")
spark.read.parquet(gold_path).show()

# Ekspor ke CSV Tunggal untuk BI Dashboard (Tableau, PowerBI, dll.)
gold_export_path = "lakehouse/exports/gold_export"
gold_agg.coalesce(1).write.mode("overwrite").option("header", "true").csv(gold_export_path)
print(f"Ekspor Gold ke CSV tunggal berhasil di: {gold_export_path}")

---
### **5. Panduan Operasional & Otomatisasi Pipeline Medallion (Jobs & Workflow)**

Panduan arsitektur operasional untuk menjalankan pipeline ini secara otomatis dalam lingkungan produksi (seperti Databricks Workflows / Apache Airflow).

#### **1. Pembuatan Job & Aturan Dependensi (DAG)**
* **Task 1: Ingest_Bronze**
  * Menjalankan notebook `01_EDA_dan_Bronze_Layer.ipynb`.
  * Tanpa dependensi upstream (root task).
* **Task 2: Clean_Silver**
  * Menjalankan notebook `02_Silver_Layer.ipynb`.
  * Dependensi: Berhasil menyelesaikan **Task 1: Ingest_Bronze**.
* **Task 3: Aggregate_Gold**
  * Menjalankan notebook `03_Gold_Layer.ipynb`.
  * Dependensi: Berhasil menyelesaikan **Task 2: Clean_Silver**.
* **Aturan Kegagalan:** Jika salah satu tahap gagal (misal Bronze), seluruh tahap hilir (Silver dan Gold) otomatis dibatalkan (*skipped* / *failed*).

#### **2. Parameterisasi & Kebijakan Retry**
* **Parameters:**
  * `source_path`: `bank_marketing.csv` (dinamis jika ada batch baru).
  * `batch_date`: Gunakan ekspresi dinamis seperti `{{now()}}` atau format tanggal harian.
* **Compute:** Pilih cluster single-node dengan runtime Spark yang sesuai (misal Databricks Runtime LTS).
* **Retry & Timeout:**
  * **Max Retries:** `2` kali percobaan ulang jika terjadi kegagalan jaringan atau timeout.
  * **Retry Interval:** `5` menit jeda antar percobaan.
  * **Timeout:** Maksimum `30` menit per task untuk mencegah pemborosan biaya compute.

#### **3. Penjadwalan (Scheduling)**
* **Metode:** Menggunakan Trigger Jadwal (Schedule / Cron) atau Event-based (File Arrival).
* **Frekuensi:** Harian (*Daily*) pada pukul `01:00 UTC` (disesuaikan dengan siklus penutupan harian perbankan).
* **Penonaktifan:** Jadwal dinonaktifkan (*paused*) setelah proses batch selesai untuk menghindari konsumsi biaya yang tidak perlu.

#### **4. Monitoring & Analisis Kegagalan**
* **Verifikasi Run:** Setelah dijalankan secara manual via tombol *Run Now*, periksa visualisasi grafik DAG (Directed Acyclic Graph) untuk memastikan semua tahapan berwarna hijau.
* **Analisis Log & Kegagalan:**
  * Jika task gagal, buka menu **Runs** -> klik task yang merah -> buka tab **Standard Error / Logs**.
  * Cari baris kesalahan yang paling bawah untuk menemukan akar masalah (seperti *file not found* atau *schema mismatch*).

#### **5. Sifat Incremental (Penanganan Batch Baru)**
* **Aturan Overwrite/Append:**
  * Pada **Bronze**, gunakan metode `.mode("overwrite")` untuk skema pengulangan file yang sama, atau gunakan `.mode("append")` dengan partisi berdasarkan tanggal batch jika ingin menyimpan histori lengkap.
  * Pada **Silver**, fungsi `.dropDuplicates()` memastikan tidak ada baris ganda yang masuk meskipun file yang sama diproses ulang.
  * Pada **Gold**, tabel agregasi di-overwrite dengan data paling mutakhir untuk menyajikan single source of truth bagi dashboard pelaporan.

In [ ]:
print("Ekspor selesai dan terintegrasi langsung dalam eksekusi di masing-masing layer (Bronze, Silver, Gold) ke dalam folder 'lakehouse/exports/'.")
print("Pipeline Medallion Bank Marketing siap dijalankan secara modular!")